# PUF256 同 bitmap 雙板驗證 — 20200

使用原始已通過時序檢查的 bitstream。共同 bitmap 僅依兩板訓練資料的穩定度與 margin 選擇；凍結後才做新的 holdout、硬體註冊與重建。每板使用自身 helper。

載入 overlay 會取代目前 PL 設計；本 Notebook 只對指定借用板操作。

In [1]:
from pathlib import Path
import json, hashlib
root = Path.cwd()
for manifest_name in ['MANIFEST.json', 'CROSS_MANIFEST.json']:
    manifest = json.loads((root / manifest_name).read_text())
    for name, record in manifest['files'].items():
        if name.endswith('.ipynb'): continue
        data = (root / name).read_bytes()
        assert len(data) == record['bytes']
        assert hashlib.sha256(data).hexdigest() == record['sha256'], name
print('Base overlay and frozen cross-board inputs verified')


Base overlay and frozen cross-board inputs verified


In [2]:
from cross_board import run_fixed
summary = run_fixed('20200', holdout_reads=32, reconstruction_runs=100, reload_runs=10)
assert summary['passed']


20200 fixed-bitmap holdout 8 / 32
20200 fixed-bitmap holdout 16 / 32
20200 fixed-bitmap holdout 24 / 32
20200 fixed-bitmap holdout 32 / 32
20200 common-bitmap reconstructions 10 / 100


RuntimeError: Fixed-bitmap reconstruction failed or changed identity

## 匯出證據

身份碼來自實際 PL 暫存器，CPU 只做輸入、量測記錄與實驗比對。結果不是跨晶片母體唯一性或熵的估計。

In [ ]:
import zipfile
from IPython.display import display, FileLink
latest = sorted(p for p in root.glob('common_20200_*') if p.is_dir())[-1]
print((latest / 'summary.json').read_text())
archive = root / (latest.name + '.zip')
with zipfile.ZipFile(str(archive), 'w', zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(latest.iterdir()):
        if path.is_file(): bundle.write(str(path), latest.name + '/' + path.name)
display(FileLink(archive.name))


In [3]:
from pynq import Overlay
from puf256 import Puf256
failed_dir = sorted(p for p in root.glob('common_20200_*') if p.is_dir())[-1]
failed_trials = json.loads((failed_dir / 'reconstructions.json').read_text())
diag_overlay = Overlay('puf256.bit', download=False)
diag_device = Puf256(diag_overlay.puf256_0)
failed_pair = diag_device.ip.read(0x2c)
print('FAILED_DIR', failed_dir.name)
print('LAST_TRIALS', json.dumps(failed_trials[-3:], indent=2))
print('FAILED_PAIR', failed_pair)
import zipfile
archive = root / (failed_dir.name + '_FAILED.zip')
with zipfile.ZipFile(str(archive), 'w', zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(failed_dir.iterdir()):
        if path.is_file(): bundle.write(str(path), failed_dir.name + '/' + path.name)


FAILED_DIR common_20200_20261113_000917
LAST_TRIALS [
  {
    "corrected_words": 0,
    "index": 16,
    "key_sha256": "a1c6d8e9b87edbc8b9807f64846520dae17015b81a6905e09300198287ab9b51",
    "match": true,
    "sequence": 18,
    "stage": "repeat",
    "status": 0
  },
  {
    "corrected_words": 0,
    "index": 17,
    "key_sha256": "a1c6d8e9b87edbc8b9807f64846520dae17015b81a6905e09300198287ab9b51",
    "match": true,
    "sequence": 19,
    "stage": "repeat",
    "status": 0
  },
  {
    "corrected_words": 0,
    "index": 18,
    "key_sha256": null,
    "match": false,
    "sequence": 20,
    "stage": "repeat",
    "status": 4
  }
]
FAILED_PAIR 25


In [4]:
import csv
from collections import Counter
probe_rows = []
for n in range(256):
    row = diag_device.measure(failed_pair)
    row['sample'] = n
    probe_rows.append(row)
with (failed_dir / 'failed_pair_probe.csv').open('w', newline='') as stream:
    writer = csv.DictWriter(stream, fieldnames=list(probe_rows[0]))
    writer.writeheader(); writer.writerows(probe_rows)
probe_result = dict(pair=failed_pair, samples=len(probe_rows), count_gap_histogram=dict(Counter(str(abs(r['delta'])) for r in probe_rows)), weak_samples=sum(r['status'] != 0 or abs(r['delta']) < 8 for r in probe_rows), last_failed_transaction=failed_trials[-1])
(failed_dir / 'failure_diagnosis.json').write_text(json.dumps(probe_result, indent=2))
print(json.dumps(probe_result, indent=2))
with zipfile.ZipFile(str(archive), 'w', zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(failed_dir.iterdir()):
        if path.is_file(): bundle.write(str(path), failed_dir.name + '/' + path.name)


{
  "pair": 25,
  "samples": 256,
  "count_gap_histogram": {
    "14": 5,
    "10": 4,
    "13": 4,
    "12": 4,
    "11": 3,
    "17": 35,
    "18": 35,
    "15": 7,
    "16": 10,
    "19": 33,
    "20": 25,
    "25": 5,
    "21": 34,
    "23": 12,
    "22": 22,
    "24": 12,
    "28": 3,
    "26": 3
  },
  "weak_samples": 0,
  "last_failed_transaction": {
    "corrected_words": 0,
    "index": 18,
    "key_sha256": null,
    "match": false,
    "sequence": 20,
    "stage": "repeat",
    "status": 4
  }
}


In [5]:
from characterize_common import resume_characterization
characterized_dir = Path(resume_characterization(str(failed_dir)))
characterized_summary = json.loads((characterized_dir / 'summary.json').read_text())
assert characterized_summary['completed']
assert characterized_summary['all_accepted_keys_match']
archive_characterized = root / (characterized_dir.name + '.zip')
with zipfile.ZipFile(str(archive_characterized), 'w', zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(characterized_dir.iterdir()):
        if path.is_file(): bundle.write(str(path), characterized_dir.name + '/' + path.name)
print('CHARACTERIZATION_ARCHIVE', archive_characterized.name)


20200 fixed scheduled attempts 20 / 100
20200 fixed scheduled attempts 30 / 100
20200 fixed scheduled attempts 40 / 100
20200 fixed scheduled attempts 50 / 100
20200 fixed scheduled attempts 60 / 100
20200 fixed scheduled attempts 70 / 100
20200 fixed scheduled attempts 80 / 100
20200 fixed scheduled attempts 90 / 100
20200 fixed scheduled attempts 100 / 100
20200 characterized reload attempt 1 / 10
20200 characterized reload attempt 2 / 10
20200 characterized reload attempt 3 / 10
20200 characterized reload attempt 4 / 10
20200 characterized reload attempt 5 / 10
20200 characterized reload attempt 6 / 10
20200 characterized reload attempt 7 / 10
20200 characterized reload attempt 8 / 10
20200 characterized reload attempt 9 / 10
20200 characterized reload attempt 10 / 10
{
  "completed": true,
  "passed": false,
  "strict_pass": false,
  "all_accepted_keys_match": true,
  "board_label": "20200",
  "bitmap_hex": "feebfe3ffff7fbedfdfbe7fbbfbbeced7ffaf1fbbc37836dfbffbb7ff4ffeafd97bfffbdfb